# Directed Biological Analyses of Vitamin D-Related Signatures

This notebook presents the **core biological analyses** of transcriptomic responses to vitamin D and its analogs, based on the LINCS L1000 dataset.  
While the previous notebooks focused on data preparation, quality control, and exploratory analysis, here we shift to **hypothesis-driven investigations**.

**Objectives of this notebook:**
- Quantify dose–response effects at the gene level.  
- Identify eligible *high vs. low dose* contrasts across compounds and cell lines.  
- Perform gene set and pathway enrichment analyses.  
- Compare analogs to distinguish **shared effects** from **compound-specific responses**.  

In [ ]:
# Core scientific stack
import os
import pandas as pd
import numpy as np

# Statistics and modeling
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities (for scaling or decomposition if needed)
from sklearn.preprocessing import StandardScaler

# Configure plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")
%matplotlib inline

## Data Setup

We begin by loading all project data tables into memory:  
- **Expression matrix** (genes × signatures).  
- **Signature metadata** (perturbation, dose, cell line, etc.).  
- **Compound metadata** (compound-level annotations).  
- **Cell line metadata** (cell type, lineage, disease).  
- **Gene metadata** (landmark vs. inferred, gene symbols).  

Having all tables available ensures that downstream analyses can seamlessly combine expression values with their biological and experimental context.


In [ ]:
# Define data directory and file paths
DATA_DIR = "../data/exports"

PATHS = {
    "exp":   f"{DATA_DIR}/expression_matrix_clean.parquet",   # expression matrix
    "sig":   f"{DATA_DIR}/signature_metadata_clean.csv",      # signature metadata
    "comp":  f"{DATA_DIR}/subset_compounds_meta.csv",         # compounds
    "cells": f"{DATA_DIR}/subset_cell_lines_meta.csv",        # cell lines
    "genes": f"{DATA_DIR}/subset_genes_meta.csv",             # genes
}

# Load all tables into memory
exp_matrix = pd.read_parquet(PATHS["exp"])
metadata   = pd.read_csv(PATHS["sig"])
compounds  = pd.read_csv(PATHS["comp"])
cell_lines = pd.read_csv(PATHS["cells"])
gene_info  = pd.read_csv(PATHS["genes"])

# Quick overview of dimensions
print(f"Expression matrix: {exp_matrix.shape[0]} genes × {exp_matrix.shape[1]} signatures")
print(f"Metadata rows:     {len(metadata)}")
print(f"Compounds:         {len(compounds)}")
print(f"Cell lines:        {len(cell_lines)}")
print(f"Genes:             {len(gene_info)}")


### Data Setup Conclusion

All data tables were successfully loaded:  
- Expression matrix with 12,328 genes × 258 signatures  
- 258 metadata entries  
- 12 compounds, 5 cell lines, and 12,328 genes  

The dataset is ready for downstream analyses.

---

## Integrity Gatekeeper

Before performing directed analyses, we run a minimal integrity check to ensure that the expression matrix and metadata are fully aligned and free of basic issues.  
This step verifies:  
- Consistent signature identifiers across tables  
- No missing values or zero-variance features  
- Dose information available and usable  


In [ ]:
def minimal_gatekeeper(exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=None):
    # Identify signature ID column in metadata
    sig_id_col = next((c for c in ["sig_id", "distil_id", "signature_id", "id"] if c in metadata.columns), None)
    if sig_id_col is None:
        raise ValueError("Signature ID column not found in metadata.")
    
    # Expression–metadata alignment (same set and order of signatures)
    exp_cols = pd.Index(map(str, exp_matrix.columns))
    meta_ids = pd.Index(metadata[sig_id_col].astype(str))
    common = exp_cols.intersection(meta_ids)
    if expected_n is not None and len(common) != expected_n:
        raise AssertionError(f"Common signatures = {len(common)} (expected {expected_n}).")
    if len(exp_cols.difference(common)) or len(meta_ids.difference(common)):
        raise AssertionError("Expression and metadata do not contain the exact same signatures.")
    meta_aligned = metadata.set_index(sig_id_col).loc[exp_cols].reset_index().rename(columns={"index": sig_id_col})
    
    # Basic integrity: NA and zero variance
    if exp_matrix.isna().any().any():
        raise AssertionError("NA values found in expression matrix.")
    if (exp_matrix.var(axis=1) == 0).any():
        raise AssertionError("Zero-variance genes detected.")
    if (exp_matrix.var(axis=0) == 0).any():
        raise AssertionError("Zero-variance signatures detected.")
    
    # Dose usability: numeric and variable within groups
    dose_col = next((c for c in meta_aligned.columns if ("dose" in c.lower()) and ("unit" not in c.lower())), None)
    if dose_col is None:
        raise AssertionError("Numeric dose column not found in metadata.")
    meta_aligned["dose_value"] = pd.to_numeric(meta_aligned[dose_col], errors="coerce")
    if meta_aligned["dose_value"].isna().any():
        raise AssertionError("Non-numeric values in dose column.")
    group_keys = [k for k in ["pert_id", "cell_id"] if k in meta_aligned.columns]
    if not group_keys:
        raise AssertionError("Missing grouping keys (pert_id/cell_id).")
    var_by_group = meta_aligned.groupby(group_keys)["dose_value"].agg(lambda x: float(np.var(x, ddof=1)) if x.notna().any() else 0.0)
    if (var_by_group == 0).all():
        raise AssertionError("No within-group dose variation; dose–response analyses are not feasible.")
    
    # Referential checks against lookup tables (lightweight)
    if "pert_id" in meta_aligned.columns and "pert_id" in compounds.columns:
        missing_comp = set(meta_aligned["pert_id"]) - set(compounds["pert_id"])
        if missing_comp:
            raise AssertionError(f"Missing compound keys in 'compounds': {len(missing_comp)}.")
    if "cell_id" in meta_aligned.columns and "cell_id" in cell_lines.columns:
        missing_cells = set(meta_aligned["cell_id"]) - set(cell_lines["cell_id"])
        if missing_cells:
            raise AssertionError(f"Missing cell IDs in 'cell_lines': {len(missing_cells)}.")
    if "gene_id" in getattr(gene_info, "columns", []):
        missing_genes = set(map(str, exp_matrix.index)) - set(map(str, gene_info["gene_id"]))
        if missing_genes:
            raise AssertionError(f"Missing gene IDs in 'gene_info': {len(missing_genes)}.")
    
    summary = {
        "signatures": len(common),
        "genes": exp_matrix.shape[0],
        "dose_col": dose_col,
        "dose_min": float(meta_aligned["dose_value"].min()),
        "dose_max": float(meta_aligned["dose_value"].max()),
        "groups_with_variation": int((var_by_group > 0).sum()),
    }
    return meta_aligned, summary

# Run gatekeeper (expecting 258 signatures based on previous step)
metadata_aligned, gate_summary = minimal_gatekeeper(
    exp_matrix, metadata, compounds, cell_lines, gene_info, expected_n=258
)

print("Gatekeeper summary:", gate_summary)


### Integrity Gatekeeper Conclusion

- 258 signatures aligned with the expression matrix  
- 12,328 genes retained  
- Dose column detected: `pert_dose` (range ≈ 0.01 – 10 µM)  
- 35 compound–cell groups show within-group dose variation  

The dataset passes all minimal integrity checks and is suitable for dose–response analyses.

---

### Dose–Response Setup

We now prepare the variables needed to model dose–response effects.  
For each signature, we create:  
- A **numeric dose value**  
- A **log10-transformed dose** for modeling trends  
- A simple **high vs. low bin** (median split within each compound–cell group)  

These variables provide the foundation for testing monotonic effects and for constructing contrasts between dose levels.


In [ ]:
# Check the global distribution of numeric doses
unique_doses = np.sort(metadata_aligned["dose_value"].unique())
median_dose = metadata_aligned["dose_value"].median()

print("Unique doses available:", unique_doses)
print("Global median dose:", median_dose)

In [ ]:
# Add numeric and log-transformed dose values
metadata_aligned["dose_value"] = pd.to_numeric(metadata_aligned["pert_dose"], errors="coerce")
metadata_aligned["log_dose"] = np.log10(metadata_aligned["dose_value"].clip(lower=1e-6))

# Create high/low bins by median split within each (compound × cell) group
def median_bin(series):
    if series.nunique() < 2:
        return pd.Series(["insufficient"] * len(series), index=series.index)
    median = series.median()
    return pd.Series(np.where(series > median, "high", "low"), index=series.index)

metadata_aligned["dose_bin"] = (
    metadata_aligned.groupby(["pert_id", "cell_id"])["dose_value"]
    .transform(median_bin)
)

# Merge compound names for readability
metadata_aligned = metadata_aligned.merge(
    compounds[["pert_id", "cmap_name"]],
    on="pert_id",
    how="left"
)

# Quick preview with compound names
metadata_aligned[["cmap_name", "pert_id", "cell_id", "dose_value", "log_dose", "dose_bin"]].head()


### Dose–Response Setup — Conclusion

- Dose variables created successfully: `dose_value`, `log_dose` (base-10), and `dose_bin` (median split within compound × cell).  
- Compound names (`cmap_name`) merged for readability (e.g., calcipotriol, calcitriol, ercalcitriol, tacalcitol).  
- Available doses in this subset span the expected log scale (0.1, 1.0, 10.0 µM).  
- The global median dose is **1.0 µM**, meaning that the median split used here effectively corresponds to a fixed threshold at 1 µM.  

These variables are ready to support both monotonic dose–response tests and binary high–low comparisons.
 
---

# Directed Biological Analyses of Vitamin D-Related Signatures
 
## Enhanced Analysis with Robust Statistical Methods
 
This notebook performs **core biological analysis** of vitamin D analog transcriptomic responses with improved statistical rigor and visualization.


## Data Setup

We first import the required Python libraries and load all project data tables (expression matrix, signature metadata, compounds, cell lines, and genes).  
This ensures a consistent environment for the directed analyses.


In [ ]:
# --- IMPORTS LIBRARIES ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, f_oneway
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
%matplotlib inline

In [ ]:
# --- DATA LOADING WITH ERROR HANDLING ---
DATA_DIR = "../data/exports"    

PATHS = {
    "exp":      f"{DATA_DIR}/expression_matrix_clean.parquet",
    "sig":      f"{DATA_DIR}/signature_metadata_clean.csv",
    "comp":     f"{DATA_DIR}/subset_compounds_meta.csv",
    "cells":    f"{DATA_DIR}/subset_cell_lines_meta.csv",
    "genes":    f"{DATA_DIR}/subset_genes_meta.csv",
}

# Validate all files exist before loading
missing_files = []
for k, p in PATHS.items():
    if not os.path.exists(p):
        missing_files.append((k, p))
        print(f"⚠️ Archivo no encontrado: {k} -> {p}")

if missing_files:
    raise FileNotFoundError(f"Missing critical files: {missing_files}")

# Load with progress indication
print("Loading data tables...")
data_objects = {}
try:
    data_objects['exp'] = pd.read_parquet(PATHS["exp"])
    data_objects['sig'] = pd.read_csv(PATHS["sig"])
    data_objects['comp'] = pd.read_csv(PATHS["comp"])
    data_objects['cells'] = pd.read_csv(PATHS["cells"])
    data_objects['genes'] = pd.read_csv(PATHS["genes"])
    print("✓ All data loaded successfully")
except Exception as e:
    raise Exception(f"Error loading data: {str(e)}")

# Assign with clearer naming
exp_matrix = data_objects['exp']
metadata = data_objects['sig']
compounds = data_objects['comp']
cell_lines = data_objects['cells']
gene_info = data_objects['genes']


In [ ]:
# --- DATA INTEGRITY CHECK ---
def validate_data_integrity(exp_matrix, metadata, compounds, cell_lines, gene_info):
    """Comprehensive data validation with detailed reporting"""
    
    results = {}
    
    # 1) Signature alignment
    sig_id_col = next((c for c in metadata.columns if c.lower() in ['sig_id', 'distil_id', 'signature_id', 'id']), None)
    if not sig_id_col:
        raise ValueError("No signature ID column found in metadata")
    
    results['sig_id_col'] = sig_id_col
    
    # Convert to string for consistent comparison
    exp_cols = set(map(str, exp_matrix.columns))
    meta_sigs = set(map(str, metadata[sig_id_col].dropna()))
    
    results['common_signatures'] = exp_cols.intersection(meta_sigs)
    results['exp_only'] = exp_cols - meta_sigs
    results['meta_only'] = meta_sigs - exp_cols
    
    # 2) Data quality metrics
    results['exp_na_count'] = exp_matrix.isna().sum().sum()
    results['exp_zero_var_genes'] = (exp_matrix.var(axis=1) == 0).sum()
    results['exp_zero_var_sigs'] = (exp_matrix.var(axis=0) == 0).sum()
    
    # 3) Cross-table consistency
    if 'pert_id' in metadata.columns and 'pert_id' in compounds.columns:
        results['missing_compounds'] = set(metadata['pert_id']) - set(compounds['pert_id'])
    
    if 'cell_id' in metadata.columns and 'cell_id' in cell_lines.columns:
        results['missing_cell_lines'] = set(metadata['cell_id']) - set(cell_lines['cell_id'])
    
    if 'gene_id' in gene_info.columns:
        results['missing_genes'] = set(exp_matrix.index.astype(str)) - set(gene_info['gene_id'].astype(str))
    
    return results, sig_id_col

# Run validation
validation_results, sig_id_col = validate_data_integrity(exp_matrix, metadata, compounds, cell_lines, gene_info)

# Filter to common signatures
common_sigs = validation_results['common_signatures']
metadata_filtered = metadata[metadata[sig_id_col].astype(str).isin(common_sigs)].copy()
exp_filtered = exp_matrix[metadata_filtered[sig_id_col].astype(str).tolist()]

print(f"""
=== ENHANCED INTEGRITY REPORT ===
Common signatures: {len(common_sigs)}
Signatures only in expression: {len(validation_results['exp_only'])}
Signatures only in metadata: {len(validation_results['meta_only'])}
NA values: {validation_results['exp_na_count']}
Zero-variance genes: {validation_results['exp_zero_var_genes']}
Zero-variance signatures: {validation_results['exp_zero_var_sigs']}
Missing compounds: {len(validation_results.get('missing_compounds', []))}
Missing cell lines: {len(validation_results.get('missing_cell_lines', []))}
Missing genes: {len(validation_results.get('missing_genes', []))}
""")

In [ ]:
# --- IMPROVED DOSE-RESPONSE ANALYSIS ---
def prepare_dose_analysis(metadata, compounds):
    """Enhanced dose response preparation with multiple binning strategies"""
    
    # Identify dose column
    dose_col = next((c for c in metadata.columns 
                    if 'dose' in c.lower() and 'unit' not in c.lower()), None)
    if not dose_col:
        raise ValueError("No dose column found in metadata")
    
    # Create working copy
    df = metadata.copy()
    
    # Convert to numeric dose
    df['dose_numeric'] = pd.to_numeric(df[dose_col], errors='coerce')
    df['log_dose'] = np.log10(df['dose_numeric'].clip(lower=1e-10))
    
    # Remove signatures without valid dose
    df = df.dropna(subset=['dose_numeric'])
    
    # Multiple binning strategies
    group_cols = []
    if 'pert_id' in df.columns: group_cols.append('pert_id')
    if 'cell_id' in df.columns: group_cols.append('cell_id')
    
    if not group_cols:
        raise ValueError("Need pert_id or cell_id for grouping")
    
    # Strategy 1: Median split within groups
    def median_bin(series):
        if series.nunique() < 2:
            return pd.Series(['insufficient_data'] * len(series), index=series.index)
        med = series.median()
        return pd.Series(np.where(series > med, 'high', 'low'), index=series.index)
    
    df['dose_bin_median'] = df.groupby(group_cols)['dose_numeric'].transform(median_bin)
    
    # Strategy 2: Quartile-based bins
    def quartile_bin(series):
        if series.nunique() < 4:
            return pd.Series(['insufficient_data'] * len(series), index=series.index)
        q1, q3 = series.quantile([0.25, 0.75])
        bins = [-np.inf, q1, q3, np.inf]
        labels = ['low', 'medium', 'high']
        return pd.cut(series, bins=bins, labels=labels)
    
    df['dose_bin_quartile'] = df.groupby(group_cols)['dose_numeric'].transform(quartile_bin)
    
    # Strategy 3: Absolute dose thresholds (compound-specific)
    compound_dose_info = df.groupby('pert_id')['dose_numeric'].agg(['min', 'median', 'max']).reset_index()
    df = df.merge(compound_dose_info, on='pert_id', suffixes=('', '_compound'))
    
    # Add compound names for readability
    if 'pert_id' in compounds.columns and 'cmap_name' in compounds.columns:
        df = df.merge(compounds[['pert_id', 'cmap_name']], on='pert_id', how='left')
    
    return df, dose_col

# Prepare dose analysis
metadata_enriched, dose_col = prepare_dose_analysis(metadata_filtered, compounds)


In [ ]:
# --- ELIGIBLE CONTRASTS IDENTIFICATION ---
def find_eligible_contrasts(metadata, min_samples=3):
    """Find valid dose-response contrasts with statistical power consideration"""
    
    contrasts = []
    
    # Group by compound and cell line
    grouping = ['pert_id', 'cell_id'] if 'cell_id' in metadata.columns else ['pert_id']
    
    for name, group in metadata.groupby(grouping):
        # Check if we have both high and low doses
        bin_counts = group['dose_bin_median'].value_counts()
        
        if 'high' in bin_counts and 'low' in bin_counts:
            n_high = bin_counts['high']
            n_low = bin_counts['low']
            
            # Only consider contrasts with sufficient samples
            if n_high >= min_samples and n_low >= min_samples:
                contrast_info = {
                    'pert_id': name[0] if isinstance(name, tuple) else name,
                    'cell_id': name[1] if isinstance(name, tuple) else 'all',
                    'n_high': n_high,
                    'n_low': n_low,
                    'total_samples': len(group),
                    'dose_range': group['dose_numeric'].max() / group['dose_numeric'].min(),
                    'mean_dose_high': group[group['dose_bin_median'] == 'high']['dose_numeric'].mean(),
                    'mean_dose_low': group[group['dose_bin_median'] == 'low']['dose_numeric'].mean()
                }
                contrasts.append(contrast_info)
    
    contrast_df = pd.DataFrame(contrasts)
    
    # Add compound names
    if 'cmap_name' in metadata.columns:
        name_map = metadata[['pert_id', 'cmap_name']].drop_duplicates().set_index('pert_id')['cmap_name']
        contrast_df['compound_name'] = contrast_df['pert_id'].map(name_map)
    
    return contrast_df.sort_values('dose_range', ascending=False)

# Find eligible contrasts
eligible_contrasts = find_eligible_contrasts(metadata_enriched, min_samples=3)

print(f"Found {len(eligible_contrasts)} eligible dose-response contrasts")
print("\nTop contrasts by dose range:")
print(eligible_contrasts.head(10).to_string(index=False))

In [ ]:
# --- VISUALIZATION OF DOSE DISTRIBUTIONS ---
def plot_dose_distributions(metadata, eligible_contrasts, top_n=10):
    """Visualize dose distributions for top contrasts"""
    
    top_contrasts = eligible_contrasts.head(top_n)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    # Plot 1: Dose range by compound
    sns.barplot(data=top_contrasts, x='compound_name', y='dose_range', ax=axes[0])
    axes[0].set_title('Dose Range (Max/Min) by Compound')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Plot 2: Sample size distribution
    sns.scatterplot(data=top_contrasts, x='dose_range', y='total_samples', 
                   size='n_high', hue='compound_name', ax=axes[1])
    axes[1].set_title('Sample Size vs. Dose Range')
    
    # Plot 3: Distribution of doses for top compound
    if not top_contrasts.empty:
        top_compound = top_contrasts.iloc[0]['pert_id']
        compound_data = metadata[metadata['pert_id'] == top_compound]
        
        sns.violinplot(data=compound_data, x='cell_id', y='log_dose', 
                      hue='dose_bin_median', ax=axes[2])
        axes[2].set_title(f'Dose Distribution for {top_contrasts.iloc[0]["compound_name"]}')
        axes[2].tick_params(axis='x', rotation=45)
    
    # Plot 4: Number of contrasts per compound
    contrast_counts = eligible_contrasts['compound_name'].value_counts()
    sns.barplot(x=contrast_counts.index, y=contrast_counts.values, ax=axes[3])
    axes[3].set_title('Number of Contrasts per Compound')
    axes[3].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

plot_dose_distributions(metadata_enriched, eligible_contrasts)

# --- NEXT STEPS PREPARATION ---
print("""
Ready for advanced analyses:
1. Differential expression analysis for each contrast
2. Pathway enrichment analysis
3. Time-course analysis
4. Compound-specific effects
5. Cell-type specific responses

Use eligible_contrasts DataFrame to iterate through valid comparisons.
""")

### Dose–Response Modeling

We quantify dose-dependent effects at the gene level.  
For each compound (within each cell line when applicable), we test whether expression changes monotonically with **log10(dose)** using (i) a linear model on z-scores and (ii) a rank-based correlation as a robustness check. Results include per-gene statistics (effect size, p-value, FDR), volcano plots, and ranked lists for enrichment.
